# YOLO Medical Detection Walkthrough

This notebook is a guided companion to the object detection chapter scripts. Use it to inspect labels, prepare dry-run YOLO commands, summarize experiment logs, and assemble the evidence needed for the homework report. Keep the `.py` files as the source of truth for repeatable command-line runs and smoke checks.

## Reader Preflight

Before running this notebook as public companion code:

- Start with the dependency check or smoke path. Real training, generation, or evaluation cells are usually guarded by flags such as `RUN_* = False`.
- Confirm dataset and model access, license or terms, and local paths before enabling external downloads or long runs.
- Keep secrets out of notebook cells. If a token is required, load it from the environment or `.env`, and keep `.env` secrets-only.
- Treat printed paths and saved JSON, CSV, and PNG artifacts as the evidence record. Rerun from a clean kernel before reporting results.

## 1. Locate the Companion Code

Run this notebook from the repository root or from `chapter_object_detection`. The setup cell imports the same workflow module used by the command-line interface.

In [ ]:
from pathlib import Path
import csv
import json
import subprocess
import sys


def find_code_dir():
    script_name = "yolo_medical_workflow.py"
    chapter_name = "chapter_object_detection"
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base, base / chapter_name, base / "code" / chapter_name):
            if (candidate / script_name).exists():
                return candidate.resolve()
    raise FileNotFoundError(
        "Run this notebook from the repository root or chapter_object_detection."
    )


CODE_DIR = find_code_dir()
REPO_ROOT = CODE_DIR.parent
WORK_DIR = Path("/tmp/adl-object-detection-notebook")
WORKFLOW = CODE_DIR / "yolo_medical_workflow.py"

sys.path.insert(0, str(CODE_DIR))
import yolo_medical_workflow as workflow

WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"Using companion code: {CODE_DIR}")
print(f"Notebook artifacts: {WORK_DIR}")

## 2. Choose Run Settings

The default settings avoid real training. Set `DATA_YAML` to a prepared YOLO dataset when one is available. If a local BCCD clone is available, set `BCCD_SOURCE_DIR` to that clone to create a small classroom fallback dataset.

In [ ]:
DATA_YAML = None  # Example: "/tmp/rsna-pneumonia-yolo/rsna-pneumonia-yolo.yaml"
BCCD_SOURCE_DIR = None  # Example: "/tmp/BCCD_Dataset"

MODEL_SMALL = "yolo26n.pt"
MODEL_LARGER = "yolo26s.pt"
EPOCHS = 100
BATCH = "auto"
SEED = 7

RUN_TRAINING = False
RUN_PREDICTION = False
PREDICT_SOURCE = None  # Example: "/tmp/rsna-pneumonia-yolo/images/test"
PREDICT_MODEL = None  # Example: "runs/object-detection/baseline/weights/best.pt"

## 3. Shared Helpers

These helpers call the companion workflow through the same CLI surface documented in the chapter README.

In [ ]:
def run_workflow(*args, check=True):
    cmd = [sys.executable, str(WORKFLOW), *map(str, args)]
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=CODE_DIR, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"workflow command failed with exit code {result.returncode}")
    return result


def read_json(path):
    path = Path(path)
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def optional_path(value):
    if value in (None, ""):
        return None
    path = Path(value).expanduser().resolve()
    return path if path.exists() else None


def show_csv(path, limit=10):
    path = Path(path)
    if not path.exists():
        print(f"Missing CSV: {path}")
        return []
    with path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    for row in rows[:limit]:
        print(row)
    if len(rows) > limit:
        print(f"... {len(rows) - limit} more row(s)")
    return rows

## 4. Record the Environment

Package versions belong in the report even when the notebook is only used for a dry run.

In [ ]:
ENV_DIR = WORK_DIR / "environment"
run_workflow("env-report", "--output-dir", ENV_DIR)
environment = read_json(ENV_DIR / "environment.json")
environment

## 5. Create a Dataset Template

Use the template when the dataset has not been prepared yet. Replace the root path and class names with the real dataset contract before training.

In [ ]:
TEMPLATE_DIR = WORK_DIR / "template"
run_workflow("make-template", "--output-dir", TEMPLATE_DIR)
print((TEMPLATE_DIR / "medical-detection.yaml").read_text(encoding="utf-8"))

## 6. Optional BCCD Fallback Conversion

BCCD is a microscopy fallback for learning the mechanics of YOLO-format detection. It should not be described as chest X-ray evidence.

In [ ]:
BCCD_OUTPUT_DIR = WORK_DIR / "bccd-yolo"
bccd_yaml = BCCD_OUTPUT_DIR / "bccd-medical-detection.yaml"
bccd_source = optional_path(BCCD_SOURCE_DIR)

if bccd_source is None:
    print("Set BCCD_SOURCE_DIR to a local BCCD clone to run this conversion.")
    print("Example: git clone https://github.com/Shenggan/BCCD_Dataset.git /tmp/BCCD_Dataset")
else:
    run_workflow(
        "prepare-bccd",
        "--source-dir",
        bccd_source,
        "--output-dir",
        BCCD_OUTPUT_DIR,
    )
    print(f"Prepared BCCD YAML: {bccd_yaml}")

## 7. Run the Lightweight Smoke Path

The smoke path validates label parsing, BCCD conversion mechanics, dry-run command writing, result summarization, and metric plotting without installing Ultralytics or training a detector.

In [ ]:
SMOKE_DIR = WORK_DIR / "smoke"
run_workflow("smoke", "--output-dir", SMOKE_DIR)
smoke_summary = read_json(SMOKE_DIR / "smoke_summary.json")
smoke_summary

## 8. Select the Active Dataset

The notebook prefers an explicitly configured dataset, then the converted BCCD fallback, then the synthetic smoke dataset. Real training should use a real dataset, not the smoke dataset.

In [ ]:
configured_yaml = optional_path(DATA_YAML)
if configured_yaml is not None:
    active_data_yaml = configured_yaml
elif bccd_yaml.exists():
    active_data_yaml = bccd_yaml.resolve()
else:
    active_data_yaml = (SMOKE_DIR / "smoke-medical-detection.yaml").resolve()

print(f"Active dataset YAML: {active_data_yaml}")
print(Path(active_data_yaml).read_text(encoding="utf-8"))

## 9. Inspect Labels Before Training

Run label inspection before any training. The homework expects at least 20 labeled training images to be inspected when a real dataset is available.

In [ ]:
INSPECTION_DIR = WORK_DIR / "label-inspection"
run_workflow(
    "inspect-labels",
    "--data-yaml",
    active_data_yaml,
    "--split",
    "train",
    "--output-dir",
    INSPECTION_DIR,
    "--max-files",
    20,
    "--example-rows",
    20,
)
inspection = read_json(INSPECTION_DIR / "label_inspection.json")
inspection

## 10. Visualize Labeled Examples

This cell draws YOLO boxes on a few training images when real image files are available. It skips cleanly for the synthetic smoke dataset.

In [ ]:
def find_image_for_label(image_dir, label_file):
    suffixes = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp")
    for suffix in suffixes:
        candidate = Path(image_dir) / f"{label_file.stem}{suffix}"
        if candidate.exists():
            return candidate
    return None


def draw_yolo_boxes(data_yaml, split="train", limit=6):
    try:
        from PIL import Image, ImageDraw
        import matplotlib.pyplot as plt
    except Exception as exc:
        print("Install pillow and matplotlib to visualize labels in the notebook.")
        print(exc)
        return []

    yaml_path = Path(data_yaml).expanduser().resolve()
    config = workflow.load_yaml(yaml_path)
    names = workflow.names_from_config(config)
    image_dir = workflow.split_path(config, yaml_path, split)
    label_dir = workflow.label_dir_for_split(config, yaml_path, split)
    rendered = []

    for label_file in sorted(label_dir.glob("*.txt")):
        image_path = find_image_for_label(image_dir, label_file)
        if image_path is None:
            continue
        try:
            image = Image.open(image_path).convert("RGB")
        except Exception:
            continue
        draw = ImageDraw.Draw(image)
        width, height = image.size
        for line in label_file.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            class_id, x_center, y_center, box_width, box_height = workflow.parse_label_line(
                line.strip(), len(names)
            )
            x1 = (x_center - box_width / 2) * width
            y1 = (y_center - box_height / 2) * height
            x2 = (x_center + box_width / 2) * width
            y2 = (y_center + box_height / 2) * height
            draw.rectangle((x1, y1, x2, y2), outline="red", width=3)
            draw.text((x1, max(0, y1 - 12)), names[class_id], fill="red")
        rendered.append((image_path, image))
        if len(rendered) >= limit:
            break

    if not rendered:
        print("No displayable image/label pairs found for this split.")
        return []

    cols = min(3, len(rendered))
    rows = (len(rendered) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes_list = list(getattr(axes, "flat", [axes]))
    for axis, (image_path, image) in zip(axes_list, rendered, strict=False):
        axis.imshow(image)
        axis.set_title(image_path.name)
        axis.axis("off")
    for axis in axes_list[len(rendered):]:
        axis.axis("off")
    fig.tight_layout()
    return rendered


draw_yolo_boxes(active_data_yaml, split="train", limit=6)

### Reader Checkpoint

After the previous cell runs, confirm that the printed paths, shapes, commands, or tables match the section description before moving on. If this checkpoint fails in a public-repo environment, fix dependencies, data paths, or guarded flags before starting longer runs.

## 11. Prepare Controlled YOLO Commands

Use dry-run commands first. A typical comparison changes one major factor at a time: small model at 640 pixels, larger model at 640 pixels, then the small model at 1024 pixels.

In [ ]:
RUNS_DIR = WORK_DIR / "runs" / "object-detection"
COMMAND_DIR = WORK_DIR / "commands"

experiments = [
    {"name": "baseline-yolo26n-640", "model": MODEL_SMALL, "imgsz": 640},
    {"name": "larger-yolo26s-640", "model": MODEL_LARGER, "imgsz": 640},
    {"name": "hires-yolo26n-1024", "model": MODEL_SMALL, "imgsz": 1024},
]

for experiment in experiments:
    run_workflow(
        "yolo",
        "--mode",
        "train",
        "--model",
        experiment["model"],
        "--data-yaml",
        active_data_yaml,
        "--epochs",
        EPOCHS,
        "--imgsz",
        experiment["imgsz"],
        "--batch",
        BATCH,
        "--seed",
        SEED,
        "--project",
        RUNS_DIR,
        "--name",
        experiment["name"],
        "--output-dir",
        COMMAND_DIR / experiment["name"],
        "--dry-run",
    )

## 12. Optional Training

Only run this cell after confirming the dataset, labels, package versions, and compute budget. It is disabled by default.

In [ ]:
if not RUN_TRAINING:
    print("RUN_TRAINING is False. Review the dry-run commands above before launching training.")
elif active_data_yaml == (SMOKE_DIR / "smoke-medical-detection.yaml").resolve():
    raise RuntimeError("Do not train on the synthetic smoke dataset.")
else:
    for experiment in experiments:
        run_workflow(
            "yolo",
            "--mode",
            "train",
            "--model",
            experiment["model"],
            "--data-yaml",
            active_data_yaml,
            "--epochs",
            EPOCHS,
            "--imgsz",
            experiment["imgsz"],
            "--batch",
            BATCH,
            "--seed",
            SEED,
            "--project",
            RUNS_DIR,
            "--name",
            experiment["name"],
            "--output-dir",
            COMMAND_DIR / experiment["name"],
        )

## 13. Summarize Runs and Plot Validation mAP

After training, this cell reads Ultralytics `results.csv` files under the run directory and writes a compact summary table and mAP curve.

In [ ]:
SUMMARY_DIR = WORK_DIR / "run-summary"
PLOTS_DIR = WORK_DIR / "plots"

run_workflow("summarize-runs", "--runs-dir", RUNS_DIR, "--output-dir", SUMMARY_DIR)
run_workflow(
    "plot-metrics",
    "--runs-dir",
    RUNS_DIR,
    "--output-dir",
    PLOTS_DIR,
    "--allow-missing-deps",
    "--allow-empty",
)
summary_rows = show_csv(SUMMARY_DIR / "run_summary.csv")
summary_rows

The next cell previews the validation mAP plot when the summary command created it. If no plot is present, use the printed file path to decide whether training outputs exist or the run directory is still empty.

In [ ]:
try:
    from IPython.display import Image, display
except Exception:
    Image = None
    display = None

plot_path = PLOTS_DIR / "metrics_map50.png"
if plot_path.exists() and Image is not None and display is not None:
    display(Image(filename=str(plot_path)))
else:
    print(f"No plot available yet: {plot_path}")

## 14. Optional Prediction and Latency Pass

Run prediction on a fixed image folder after selecting a final model using validation metrics. The workflow writes timing metadata that can be included in the accuracy-speed report.

In [ ]:
if not RUN_PREDICTION:
    print("RUN_PREDICTION is False. Set PREDICT_MODEL and PREDICT_SOURCE before running prediction.")
else:
    model_path = optional_path(PREDICT_MODEL)
    source_path = optional_path(PREDICT_SOURCE)
    if model_path is None or source_path is None:
        raise FileNotFoundError("PREDICT_MODEL and PREDICT_SOURCE must both point to existing paths.")
    run_workflow(
        "yolo",
        "--mode",
        "predict",
        "--model",
        model_path,
        "--source",
        source_path,
        "--imgsz",
        640,
        "--project",
        RUNS_DIR,
        "--name",
        "selected-model",
        "--output-dir",
        COMMAND_DIR / "selected-model-predict",
    )

## 15. Report Checklist

- Dataset source, license or terms, preprocessing, class list, and split policy.
- Evidence from label inspection before training, including visual examples.
- Package versions, hardware, seed, model checkpoint, image size, batch size, and epoch count.
- Validation metrics and curves for controlled experiments.
- Final one-time test result for the selected model.
- Inference latency on a fixed image folder.
- Qualitative examples: correct detections, missed objects, false positives, duplicates, and loose boxes.
- Limitations paragraph stating that the model is educational and not a clinical diagnostic system.

## Reader Report Checklist

Before using results from this notebook in a report or downstream example, record:

- The notebook name, companion script, package versions, device, seed, and any guarded flags you enabled.
- The dataset or sample-data source, license or terms, split policy, and any preprocessing or synthetic fallback used.
- The exact artifact files that support the result, preferably saved JSON, CSV, or PNG files rather than transient cell output.
- The validation evidence used for model or configuration selection, and whether any final-test cell was run exactly once.
- The main limitation of the run, such as tiny synthetic data, missing optional dependencies, short training budget, or unavailable model access.